In [38]:
from pathlib import Path
import json
import subprocess
import shlex

QUAL_SEED = 1 # [1, 7, 21, 42, 123]

REPO_ROOT = Path("..").resolve()

QUAL_ROOT = REPO_ROOT / "outputs" / f"qual_{QUAL_SEED}"
QUAL_ROOT.mkdir(parents=True, exist_ok=True)

# Change these checkpoint paths to your actual .pth files.
# I used clear placeholders based on your experiment names.
EXPERIMENTS_4 = {
    "Baseline": {
        "checkpoint": REPO_ROOT / "external" / "checkpoints" / "checkpoint-best-ham-base-weighted-nomix.pth",
        "checkpoint_model_type": "panderm",
        "use_seg_gate": False,
        # "out_dir_topk": QUAL_ROOT / "cam_baseline_gt_topk3",
        "out_dir_topk": QUAL_ROOT / f"cam_baseline_gt_topk3_seed{QUAL_SEED}",
        "out_dir_fixed": QUAL_ROOT / f"cam_baseline_fixed_mel_nv_seed{QUAL_SEED}",
    },
    # "HA + DAL": {
    #     "checkpoint": REPO_ROOT / "external" / "checkpoints" / "checkpoint-best-dal-ha.pth",
    #     "checkpoint_model_type": "panderm",
    #     "use_seg_gate": False,
    #     "out_dir_topk": QUAL_ROOT / f"cam_ha_dal_gt_topk3_seed{QUAL_SEED}",
    #     "out_dir_fixed": QUAL_ROOT / f"cam_ha_dal_fixed_mel_nv_seed{QUAL_SEED}",
    # },
    "HA": {
        "checkpoint": REPO_ROOT / "external" / "checkpoints" / "checkpoint-best-HA_lam1-base-weighted-nomix.pth",
        "checkpoint_model_type": "panderm",
        "use_seg_gate": False,
        "out_dir_topk": QUAL_ROOT / f"cam_ha_gt_topk3_seed{QUAL_SEED}",
        "out_dir_fixed": QUAL_ROOT / f"cam_ha_fixed_mel_nv_seed{QUAL_SEED}",
    },
    "SegGate": {
        "checkpoint": REPO_ROOT / "external" / "checkpoints" / "checkpoint-best-seggate.pth",
        "checkpoint_model_type": "seggate",
        "use_seg_gate": True,
        "seg_gate_bg_keep": 0.05,
        "out_dir_topk": QUAL_ROOT / f"cam_seggate_gt_topk3_seed{QUAL_SEED}",
        "out_dir_fixed": QUAL_ROOT / f"cam_seggate_fixed_mel_nv_seed{QUAL_SEED}",
    },
    "SegGate + HA": {
        "checkpoint": REPO_ROOT / "external" / "checkpoints" / "checkpoint-best-seggate-ha.pth",
        "checkpoint_model_type": "seggate",
        "use_seg_gate": True,
        "seg_gate_bg_keep": 0.05,
        "out_dir_topk": QUAL_ROOT / f"cam_seggate_ha_gt_topk3_seed{QUAL_SEED}",
        "out_dir_fixed": QUAL_ROOT / f"cam_seggate_ha_fixed_mel_nv_seed{QUAL_SEED}",
    },
}

In [39]:
# Helper to run CAM generation
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)

    if dry_run:
        return

    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def generate_qualitative_cams(
    experiments: dict,
    csv_path: Path,
    img_dir: Path,
    mode_name: str,
    compare_mode: str,
    out_dir_key: str,
    num_samples: int,
    A: str | None = None,
    B: str | None = None,
    dry_run: bool = False,
):
    for exp_name, cfg in experiments.items():
        out_dir = cfg[out_dir_key]
        out_dir.mkdir(parents=True, exist_ok=True)

        is_seggate = cfg.get("use_seg_gate", False)

        if is_seggate:
            panel_items = (
                "rgb_gt_mask,"
                "gate_weighted_gradcam_a,"
                "gate_weighted_gradcam_b,"
                "gate_weighted_map_diff,"
                "gate_weighted_finercam"
            )
        else:
            panel_items = (
                "rgb_gt_mask,"
                "gradcam_a,"
                "gradcam_b,"
                "map_diff,"
                "finercam"
            )

        cmd = [
            "python", "-m", "scripts.generate_finer_cam_panderm",
            "--csv", str(csv_path),
            "--image_col", "image_rel_path",
            "--img_dir", str(img_dir),
            "--gt_col", "gt_label",
            "--checkpoint", str(cfg["checkpoint"]),
            "--checkpoint_model_type", cfg["checkpoint_model_type"],
            "--class_preset", "ham",
            "--out_dir", str(out_dir),
            "--num_samples", str(num_samples),
            "--method", "finercam",
            "--compare_mode", compare_mode,
            "--topk_compare", "3",
            "--alpha", "0.8",
            "--panel_items", panel_items,
            "--mask_root", str(REPO_ROOT / "data" / "HAM10000"),
            "--mask_col", "mask_rel_path",
        ]

        if A is not None:
            cmd += ["--A", A]
        if B is not None:
            cmd += ["--B", B]

        if is_seggate:
            cmd += [
                "--use_seg_gate",
                "--seg_gate_bg_keep", str(cfg.get("seg_gate_bg_keep", 0.05)),
            ]

        print(f"\nRunning {mode_name}: {exp_name}")
        run_command(cmd, dry_run=dry_run)

In [40]:
# Generate Top-k GT CAM panels
# TOPK_CSV = REPO_ROOT / "data" / "HAM10000" / "ham_test_cam_qualitative_stratified_10.csv"
TOPK_CSV = REPO_ROOT / "data" / "HAM10000" / f"ham_test_cam_qualitative_stratified_10_seed{QUAL_SEED}.csv"

# Important:
# Use data/HAM10000, not data/HAM10000/images,
# because image_rel_path already contains "images/ISIC_xxx.jpg".
IMG_DIR = REPO_ROOT / "data" / "HAM10000"

generate_qualitative_cams(
    experiments=EXPERIMENTS_4,
    csv_path=TOPK_CSV,
    img_dir=IMG_DIR,
    mode_name="Top-k GT",
    compare_mode="gt_topk_non_target",
    out_dir_key="out_dir_topk",
    num_samples=10,
    dry_run=False,   # set True first if you only want to print commands
)


Running Top-k GT: Baseline

python -m scripts.generate_finer_cam_panderm --csv /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/ham_test_cam_qualitative_stratified_10_seed1.csv --image_col image_rel_path --img_dir /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --gt_col gt_label --checkpoint /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints/checkpoint-best-ham-base-weighted-nomix.pth --checkpoint_model_type panderm --class_preset ham --out_dir /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_1/cam_baseline_gt_topk3_seed1 --num_samples 10 --method finercam --compare_mode gt_topk_non_target --topk_compare 3 --alpha 0.8 --panel_items rgb_gt_mask,gradcam_a,gradcam_b,map_diff,finercam --mask_root /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --mask_col mask_rel_path
[warn] load_state_dict mismatch: missing=0, unexpected=24
  unexpected sample: ['blocks.0.attn.relative_position_bias_table', 'b

In [41]:
# # Generate fixed MEL vs NV CAM panels
# FIXED_CSV = REPO_ROOT / "data" / "HAM10000" / "ham_test_mel_nv_fixed70.csv"

# generate_qualitative_cams(
#     experiments=EXPERIMENTS_4,
#     csv_path=FIXED_CSV,
#     img_dir=IMG_DIR,
#     mode_name="Fixed MEL vs NV",
#     compare_mode="fixed",
#     A="MEL",
#     B="NV",
#     out_dir_key="out_dir_fixed",
#     num_samples=10,
#     dry_run=False,
# )

In [42]:
# Write experiment JSON configs for PDF generation
CONFIG_DIR = REPO_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

topk_pdf_config = [
    {"name": name, "folder": str(cfg["out_dir_topk"].relative_to(REPO_ROOT))}
    for name, cfg in EXPERIMENTS_4.items()
]

# fixed_pdf_config = [
#     {"name": name, "folder": str(cfg["out_dir_fixed"].relative_to(REPO_ROOT))}
#     for name, cfg in EXPERIMENTS_4.items()
# ]

# TOPK_JSON = CONFIG_DIR / "qualitative_topk3_4experiments.json"
TOPK_JSON = CONFIG_DIR / f"qualitative_topk3_4experiments_seed{QUAL_SEED}.json"
# FIXED_JSON = CONFIG_DIR / "qualitative_fixed_mel_nv_4experiments.json"

TOPK_JSON.write_text(json.dumps(topk_pdf_config, indent=2))
# FIXED_JSON.write_text(json.dumps(fixed_pdf_config, indent=2))

print("Saved:", TOPK_JSON)
# print("Saved:", FIXED_JSON)

Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/configs/qualitative_topk3_4experiments_seed1.json


In [43]:
# Build comparison PDFs
def build_qualitative_pdf(
    csv_path: Path,
    experiments_json_path: Path,
    out_pdf: Path,
    num_samples: int = 10,
    dry_run: bool = False,
):
    out_pdf.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "-m", "scripts.make_qualitative_comparison_pdf",
        "--csv", str(csv_path),
        "--image_col", "image_rel_path",
        "--gt_col", "gt_label",
        "--out_pdf", str(out_pdf),
        "--experiments_json_path", str(experiments_json_path),
        "--num_samples", str(num_samples),
        "--missing_policy", "placeholder",
    ]

    run_command(cmd, dry_run=dry_run)


build_qualitative_pdf(
    csv_path=TOPK_CSV,
    experiments_json_path=TOPK_JSON,
    # out_pdf=QUAL_ROOT / "qualitative_topk3_4experiments.pdf",
    out_pdf=QUAL_ROOT / f"qualitative_topk3_4experiments_seed{QUAL_SEED}.pdf",
    num_samples=10,
    dry_run=False,
)

# build_qualitative_pdf(
#     csv_path=FIXED_CSV,
#     experiments_json_path=FIXED_JSON,
#     out_pdf=QUAL_ROOT / "qualitative_fixed_mel_nv_4experiments.pdf",
#     num_samples=10,
#     dry_run=False,
# )


python -m scripts.make_qualitative_comparison_pdf --csv /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/ham_test_cam_qualitative_stratified_10_seed1.csv --image_col image_rel_path --gt_col gt_label --out_pdf /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_1/qualitative_topk3_4experiments_seed1.pdf --experiments_json_path /Users/choekyelnyungmartsang/Developer/master-thesis/configs/qualitative_topk3_4experiments_seed1.json --num_samples 10 --missing_policy placeholder
Saved PDF: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_1/qualitative_topk3_4experiments_seed1.pdf
Pages written: 10
